# 🎵 [제03화] 마지막 상여 — YuE 원클릭 음원 생성기 (Colab T4 최적화판)

- **시집**: 『노을빛 기억』
- **작품**: [제03화] 「마지막 상여」
- **출력 파일**: `03_마지막_상여.mp3`
- **환경**: Google Colab 무료 T4 GPU (15GB VRAM)

> 💡 **[초간단 원클릭 실행 안내]**
> 상단 메뉴에서 **[런타임] → [모두 실행]**을 누르면 자동으로 패키지 설치부터 모델 로드, 음원 생성, 그리고 브라우저로 **`03_마지막_상여.mp3` 자동 다운로드**까지 완벽하게 진행됩니다.

In [ ]:
# [1단계] GPU 하드웨어 가속기 점검 (Google Colab T4 16GB VRAM)
!nvidia-smi

import torch
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"연결된 GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM 총량: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("GPU가 감지되지 않았습니다. [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택해 주세요!")

In [ ]:
# [2단계] YuE 및 오디오 신디사이징 필수 종속성 패키지 설치
!pip install -q --upgrade pip
!pip install -q torch torchaudio transformers accelerate soundfile librosa pydub einops omegaconf sentencepiece bitsandbytes

# YuE 리포지토리 클론 및 로컬 패키지 설치
%cd /content
!rm -rf /content/YuE
!git clone https://github.com/multimodal-art-projection/YuE.git /content/YuE
%cd /content/YuE
!pip install -q -e . || true

print("✅ 필수 패키지 및 YuE 엔진 설치 완료!")

In [ ]:
# @title [3단계] 노랫말 및 사운드 프롬프트 구성 (동적 조립 모듈)
# @markdown 스토리북 스튜디오에서 생성된 노랫말과 장르 프롬프트가 자동으로 설정되어 있습니다.
# @markdown 다른 곡으로 변경하려면 아래 값을 수정하신 후 3~5단계를 다시 실행하세요.
import os

SONG_EPISODE = "제03화"  # @param {type:"string"}
SONG_TITLE = "마지막 상여"  # @param {type:"string"}
TARGET_FILENAME = "03_마지막_상여.mp3"  # @param {type:"string"}

# 음악 사운드 스타일 태그
GENRE_PROMPT = "Korean Traditional Folk Ballad, Mournful Elegy, Han, Gugak Crossover, Haegeum, Daegum solo, Gayageum, Janggu, Deep Korean Buk, Soft Strings, Low Piano, Slow 68 BPM, Joongmori rhythm, sorrowful traditional female vocal, heartbreaking"

# 완성된 노래 가사
LYRICS_CONTENT = """[Genre: Korean Traditional Folk Ballad, Mournful Elegy, Han, Gugak Crossover]
[Instruments: Haegeum, Daegum, Gayageum, Janggu, Deep Korean Buk, Soft Strings]
[Tempo: Slow, 68 BPM, Joongmori rhythm]
[Mood: Heartbreaking, Tearful, Sorrowful, Poetic]

[Intro]
(구슬픈 대금 독주로 시작되어 빗소리와 함께 해금의 처연한 선율이 얹어진다. 장구의 느린 북편 소리가 가슴을 먹먹하게 두드린다.)

[Verse 1]
목화는 어느덧 전설이 되고
모시올 풀던 날 아득하여라
겨울엔 살 덥히는 무명베를 짜고
여름엔 숨통 틔울 올을 고르며
사철 우리 어머니 쉴 날 없었네

사대가 모여 사는 종갓집 맏며느리
환갑이 지나도록 층층시하라
서방님 손목 잡고 마실 문턱조차
한 번을 나선 적이 없던 사람아

[Verse 2]
전란의 포화 속에 뼈나 찾을까
점쟁이 말대로 살아는 왔으나
마당에 쏟아내던 기침 소리엔
아직도 화약 내음 묻어 있었지

산신령님 조왕님 감사하여라
두 손 모아 눈물로 빌고 빌어도
칠순의 문턱에서 숨이 꺾이어
애비 노릇 그만두고 눈을 감았네

[Chorus]
오뉴월 장대비는 억수로 퍼붓는데
장롱 속 오십 년 묵은 모시 수의
고이 접은 슬픔을 몸에 두르고
먼 길 떠나는 임을 배웅하누나
골목길 굽이돌아 꽃상여 나가니
빗물에 상여꽃이 녹아내리네

[Bridge]
(장구와 북소리가 중중모리로 고조되며, 해금이 애간장을 끊듯 솟구친다)
어허- 넘차 달구야
어허- 넘차 달구야
어른 노릇 그 삶이 그리 버거워
자기 어머니 등진 지 석 달 만에
홀로 이 빗속을 떠나가시오

[Chorus]
오뉴월 장대비는 억수로 퍼붓는데
장롱 속 오십 년 묵은 모시 수의
고이 접은 슬픔을 몸에 두르고
먼 길 떠나는 임을 배웅하누나
골목길 굽이돌아 꽃상여 나가니
빗물에 상여꽃이 녹아내리네

[Outro]
베틀 소리 끊긴 지 반백 년 세월
달구 소리만 구슬프게 골목을 도네
녹아내린 꽃잎 따라 길을 떠나는
우리 마을의, 아…… 마지막 상여여라.
(대금의 여운과 빗소리가 천천히 잦아들며 페이드아웃)"""

# 추론용 파일 저장
os.makedirs("/content/input", exist_ok=True)
with open("/content/input/genre.txt", "w", encoding="utf-8") as f:
    f.write(GENRE_PROMPT.strip())
with open("/content/input/lyrics.txt", "w", encoding="utf-8") as f:
    f.write(LYRICS_CONTENT.strip())

print(f"✅ 3단계: [{SONG_EPISODE}] {SONG_TITLE} 가사 및 프롬프트 준비 완료! (/content/input/)")

In [ ]:
# [4단계] YuE 오디오 추론 및 완곡 음원 렌더링 (실시간 로그 출력)
import os
import glob
import shutil

os.makedirs("/content/output", exist_ok=True)
print(f"🎵 YuE 모델 로드 및 음원 추론을 시작합니다: [{SONG_EPISODE}] {SONG_TITLE}...")

# inference 디렉토리로 이동 후 추론 실행
%cd /content/YuE/inference
!python infer.py \
  --stage1_model m-a-p/YuE-s1-7B-anneal-en-cot \
  --stage2_model m-a-p/YuE-s2-1B-general \
  --genre_txt /content/input/genre.txt \
  --lyrics_txt /content/input/lyrics.txt \
  --run_n_segments 2 \
  --stage2_batch_size 4 \
  --output_dir /content/output \
  --cuda_idx 0 \
  --max_new_tokens 3000

# 생성된 음원 탐색 및 최종 파일 통일
audio_candidates = glob.glob("/content/output/**/*.mp3", recursive=True) + glob.glob("/content/output/**/*.wav", recursive=True)
final_audio = os.path.join("/content", TARGET_FILENAME)

if audio_candidates:
    src = audio_candidates[0]
    if src.endswith(".mp3"):
        shutil.copy(src, final_audio)
    else:
        !ffmpeg -y -i "{src}" "{final_audio}"
    print(f"🎉 최종 음원 준비 완료: {final_audio}")
else:
    print("⚠️ output 디렉토리에 생성된 오디오 파일이 없습니다. 위 에러 로그를 확인해 주세요.")

In [ ]:
# [5단계] 완성된 테마곡 미리듣기 및 원클릭 자동 다운로드
import os
from IPython.display import Audio, display
from google.colab import files

TARGET_AUDIO = os.path.join("/content", TARGET_FILENAME)

if os.path.exists(TARGET_AUDIO):
    print("=" * 65)
    print(f"🎶 [{SONG_EPISODE}] {SONG_TITLE} 테마곡 재생")
    print("=" * 65)
    display(Audio(TARGET_AUDIO, autoplay=True))
    
    print(f"\n📥 대표님 PC의 [다운로드] 폴더로 음원({TARGET_FILENAME}) 자동 다운로드를 시작합니다...")
    files.download(TARGET_AUDIO)
    print(f"💡 다운로드받은 '{TARGET_FILENAME}' 파일을 스토리북 스튜디오의 [음원 등록]에 등록하세요!")
else:
    print("⚠️ 음원 파일이 아직 생성되지 않았습니다. 4단계 셀의 출력을 확인해 주세요.")